# Học tăng cường áp dụng cho trò chơi TTT

## Môi trường game TTT

In [1]:
import numpy as np
import random
from copy import deepcopy
class action_space:
    def __init__(self, n):
        self.n = n
    
class observation_space:
    def __init__(self, n):
        self.shape = (n,)
class ttt:
    def __init__(self): 
        self.action_space = action_space(9)
        self.observation_space = observation_space(9)
        self.info = ""         
        self.cellcenter = {1:(-200,-200), 2:(0,-200), 3:(200,-200),
                           4:(-200,0),    5:(0,0),    6:(200,0),
                           7:(-200,200),  8:(0,200),  9:(200,200)} 
        self.reset()
        
    def sample(self):
        return random.choice(self.validinputs)   
    def reset(self):  
        self.turn = "X"
        self.rounds = 1
        self.validinputs = list(range(1, 10))
        self.occupied = {"X": [], "O": []}
        self.state = np.array([0]*9)
        self.done = False
        self.reward = 0     
        return self.state        
        
    def step(self, inp):
        inp = int(inp)
        self.occupied[self.turn].append(inp)
        self.state[inp - 1] = 1 if self.turn == "X" else -1
        self.validinputs.remove(inp) 
        
        if self.win_game():
            self.done = True
            self.reward = 1 if self.turn == "X" else -1
            self.validinputs = []
        elif self.rounds == 9:
            self.done = True
            self.reward = 0
            self.validinputs = []
        else:
            self.rounds += 1
            self.turn = "O" if self.turn == "X" else "X"             
        return self.state, self.reward, self.done, self.info
                    
    def win_game(self):
        lst = self.occupied[self.turn]
        lines = [
            [1, 2, 3], [4, 5, 6], [7, 8, 9],
            [1, 4, 7], [2, 5, 8], [3, 6, 9],
            [1, 5, 9], [3, 5, 7]
        ]
        for line in lines:
            if line[0] in lst and line[1] in lst and line[2] in lst:
                return True
        return False
print("✅ Đã khởi tạo môi trường ttt() độc lập, sẵn sàng chạy trên Kaggle!")

✅ Đã khởi tạo môi trường ttt() độc lập, sẵn sàng chạy trên Kaggle!


## Khởi tạo Q-learning values

In [2]:
# =============================================================================
# 1. KHỞI TẠO BẢNG GIÁ TRỊ Q-TABLE CHO TICTACTOE
# =============================================================================
import numpy as np

# Bảng Q-table: 
# Key   : tuple 9 phần tử biểu diễn trạng thái bàn cờ
# Value : mảng numpy (9,) lưu Q-value của 9 nước đi tương ứng (ô 1 -> ô 9)
Q_table = {}

def get_q_values(state_key):
    """Lấy vector Q-value của trạng thái state_key; nếu chưa có thì khởi tạo vector 0."""
    if state_key not in Q_table:
        Q_table[state_key] = np.zeros(9, dtype=np.float32)
    return Q_table[state_key]

# Kiểm tra thử với bàn cờ trống
empty_board_key = tuple([0] * 9)
print("✅ Đã khởi tạo cấu trúc bảng Q_table!")
print(f"- Giá trị Q-values cho bàn cờ trống ban đầu:\n  {get_q_values(empty_board_key)}")
print(f"- Tổng số trạng thái đã lưu hiện tại: {len(Q_table)}")

✅ Đã khởi tạo cấu trúc bảng Q_table!
- Giá trị Q-values cho bàn cờ trống ban đầu:
  [0. 0. 0. 0. 0. 0. 0. 0. 0.]
- Tổng số trạng thái đã lưu hiện tại: 1


## Nhận diện môi trường 

In [3]:
from copy import deepcopy
from math import sqrt, log
import pandas as pd

# --- THUẬT TOÁN PURE MCTS TỰ CHỨA ĐỂ THI ĐẤU ---
def mcts_expand(env, move):
    e = deepcopy(env)
    s, r, d, _ = e.step(move)
    return e, d, r

def mcts_simulate(env_copy, done, reward):
    if done: return reward
    while True:
        m = env_copy.sample()
        _, r, d, _ = env_copy.step(m)
        if d: return r

def mcts_backpropagate(env, move, reward, counts, wins, losses):
    counts[move] = counts.get(move, 0) + 1
    if reward == 1 and env.turn == "X": wins[move] = wins.get(move, 0) + 1
    elif reward == -1 and env.turn == "O": wins[move] = wins.get(move, 0) + 1
    elif reward == -1 and env.turn == "X": losses[move] = losses.get(move, 0) + 1
    elif reward == 1 and env.turn == "O": losses[move] = losses.get(move, 0) + 1
    return counts, wins, losses

def mcts_select(env, counts, wins, losses, temperature=1.414):
    for k in env.validinputs:
        if counts[k] == 0: return k
    N = sum(counts.values())
    scores = {k: (wins.get(k, 0) - losses.get(k, 0))/counts[k] + temperature * sqrt(log(N)/counts[k]) for k in env.validinputs}
    return max(scores, key=scores.get)

def pure_mcts_move(env, num_rollouts=60):
    if len(env.validinputs) == 1: return env.validinputs[0]
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    for _ in range(num_rollouts):
        m = mcts_select(env, counts, wins, losses)
        ec, d, r = mcts_expand(env, m)
        r = mcts_simulate(ec, d, r)
        counts, wins, losses = mcts_backpropagate(env, m, r, counts, wins, losses)
    scores = {k: (wins.get(k, 0) - losses.get(k, 0))/counts[k] if counts[k] > 0 else -float('inf') for k in counts.keys()}
    return max(scores, key=scores.get)

In [4]:
# =============================================================================
# 3. NHẬN DIỆN MÔI TRƯỜNG & CHUẨN HÓA TRẠNG THÁI (CANONICAL STATE)
# =============================================================================
def get_canonical_state(env):
    """
    Chuẩn hóa góc nhìn trạng thái:
    - Lượt quân X: quân mình là 1, đối thủ là -1.
    - Lượt quân O: đảo dấu (-state), quân mình vẫn là 1, đối thủ là -1.
    """
    turn_mult = 1 if env.turn == "X" else -1
    return tuple(env.state * turn_mult)

# Thử nghiệm nhận diện môi trường
test_env = ttt()
test_env.reset()
test_env.step(5)  # X đánh vào ô tâm (ô số 5)

print("🎮 BÀN CỜ MẪU (X vừa đánh ô 5, đến lượt O):")
print(test_env.state.reshape(3, 3)[::-1])
print(f"- Lượt đi hiện tại           : Quân '{test_env.turn}'")
print(f"- Các ô trống hợp lệ         : {test_env.validinputs}")
print(f"- Vector trạng thái gốc      : {tuple(test_env.state)}")
print(f"- Trạng thái chuẩn hóa cho O : {get_canonical_state(test_env)}")

🎮 BÀN CỜ MẪU (X vừa đánh ô 5, đến lượt O):
[[0 0 0]
 [0 1 0]
 [0 0 0]]
- Lượt đi hiện tại           : Quân 'O'
- Các ô trống hợp lệ         : [1, 2, 3, 4, 6, 7, 8, 9]
- Vector trạng thái gốc      : (0, 0, 0, 0, 1, 0, 0, 0, 0)
- Trạng thái chuẩn hóa cho O : (0, 0, 0, 0, -1, 0, 0, 0, 0)


## Khởi tạo tác nhân

Áp dụng công thức tối ưu phần thưởng của Q learning table, ta khởi tạo tác nhân tự học chơi TicTacToe và quan sát Q-value của mô hình

In [5]:
# =============================================================================
# 2. KHỞI TẠO TÁC NHÂN Q-LEARNING VỚI CHIẾN LƯỢC EPSILON-GREEDY
# =============================================================================
import random

def select_action(env, state_key, epsilon=0.1):
    """
    Lựa chọn nước đi theo chiến lược Epsilon-Greedy:
    - Với xác suất epsilon: Khám phá (Exploration) -> đi ngẫu nhiên ô hợp lệ.
    - Với xác suất 1 - epsilon: Khai thác (Exploitation) -> chọn ô có Q-value cao nhất trong các ô hợp lệ.
    """
    valid_moves = env.validinputs
    if len(valid_moves) == 0:
        return None
    if len(valid_moves) == 1:
        return valid_moves[0]
        
    # Khám phá ngẫu nhiên
    if np.random.rand() < epsilon:
        return random.choice(valid_moves)
        
    # Khai thác Q-table
    q_vals = get_q_values(state_key)
    valid_q = {m: q_vals[m - 1] for m in valid_moves}
    max_q = max(valid_q.values())
    
    # Nếu có nhiều nước đi cùng đạt max Q, chọn ngẫu nhiên giữa các nước đó
    best_moves = [m for m, val in valid_q.items() if val == max_q]
    return random.choice(best_moves)

def q_agent_move(env):
    """Quyết định nước đi thuần túy cho thi đấu (Greedy tuyệt đối, epsilon = 0.0)."""
    state_key = get_canonical_state(env)
    return select_action(env, state_key, epsilon=0.0)

print("✅ Đã khởi tạo tác nhân Q-learning với chiến lược Epsilon-Greedy thành công!")

✅ Đã khởi tạo tác nhân Q-learning với chiến lược Epsilon-Greedy thành công!


## Định nghĩa phần thưởng

In [6]:
# =============================================================================
# 4. ĐỊNH NGHĨA PHẦN THƯỞNG & HÀM CẬP NHẬT PHƯƠNG TRÌNH BELLMAN
# =============================================================================
"""
Quy ước phần thưởng (Zero-Sum Reward):
- Thắng ván cờ  : +1.0
- Thua ván cờ   : -1.0
- Hòa ván cờ    :  0.0
- Nước đi thường:  0.0

Công thức cập nhật Bellman Equation (Temporal Difference Q-learning):
Q(s, a) = Q(s, a) + lr * [ Target - Q(s, a) ]
Trong đó:
- Target nước kết thúc : Reward (+1, -1, hoặc 0)
- Target nước trung gian: Reward + gamma * max_a' Q(s', a')
"""

def update_q_value(state_key, action, target, lr):
    """Cập nhật giá trị Q cho cặp trạng thái - hành động (s, a)."""
    q_vals = get_q_values(state_key)
    current_q = q_vals[action - 1]
    q_vals[action - 1] += lr * (target - current_q)

print("✅ Đã thiết lập cấu trúc phần thưởng và hàm cập nhật Bellman Equation!")

✅ Đã thiết lập cấu trúc phần thưởng và hàm cập nhật Bellman Equation!


## Training theo Episode

```python
# the learning rate
lr=0.01
# discount rate
gamma=0.95
# parameters to control exploration
max_exp=0.9
min_exp=0.1
# maximum steps in a game
max_steps=50
# number of episodes to train Q-values
max_episode=10000
```

In [46]:
# =============================================================================
# 5. VÒNG LẶP HUẤN LUYỆN Q-LEARNING QUA CÁC EPISODE (SELF-PLAY)
# =============================================================================
import time

# Thiết lập các siêu tham số huấn luyện (theo khung tham số chương 12)
lr = 0.1               # Tốc độ học (learning rate)
gamma = 0.8            # Hệ số chiết khấu phần thưởng tương lai
max_exp = 0.9           # Mức khám phá ban đầu (90% ngẫu nhiên)
min_exp = 0.1          # Mức khám phá cuối cùng (5% ngẫu nhiên)
max_episode = 100000     # Số ván cờ huấn luyện
save_interval = 2000    # Chu kỳ báo cáo kết quả

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN Q-LEARNING QUA {max_episode} VÁN TỰ ĐẤU (SELF-PLAY)...")
t_start = time.time()

stats_x_wins = 0
stats_o_wins = 0
stats_ties = 0

# Khởi tạo môi trường bàn cờ TicTacToe
env = ttt()

for ep in range(1, max_episode + 1):
    env.reset()
    # Epsilon suy giảm tuyến tính theo tiến trình học
    epsilon = max_exp - (max_exp - min_exp) * (ep / max_episode)
    trajectory = []
    
    while True:
        state_key = get_canonical_state(env)
        player_turn = env.turn
        
        # Chọn nước đi theo Epsilon-Greedy
        action = select_action(env, state_key, epsilon)
        trajectory.append((state_key, action, player_turn))
        
        state, reward, done, _ = env.step(action)
        if done:
            break
            
    # Ghi nhận kết quả ván cờ
    if reward == 1:
        stats_x_wins += 1
    elif reward == -1:
        stats_o_wins += 1
    else:
        stats_ties += 1

    # Cập nhật Bellman lan truyền ngược theo quỹ đạo ván đấu (TD-Learning)
    final_reward_x = reward  # 1 nếu X thắng, -1 nếu O thắng, 0 nếu hòa
    
    for i in reversed(range(len(trajectory))):
        s_k, act, p_turn = trajectory[i]
        # Phần thưởng đối với người chơi ở lượt đó
        p_reward = final_reward_x if p_turn == "X" else -final_reward_x
        
        # Nước đi dẫn đến kết thúc ván
        if i >= len(trajectory) - 2:
            target = p_reward
        else:
            # Trạng thái kế tiếp của chính người chơi đó (sau khi đối thủ đã đáp trả)
            next_s_k, _, _ = trajectory[i + 2]
            next_max_q = np.max(get_q_values(next_s_k))
            target = p_reward + gamma * next_max_q
            
        update_q_value(s_k, act, target, lr)
        
    # Báo cáo tiến độ định kỳ
    if ep % save_interval == 0 or ep == max_episode:
        elapsed = time.time() - t_start
        print(f"Episode {ep:5d}/{max_episode} ({elapsed:5.1f}s) | Epsilon: {epsilon:.3f} | "
              f"Số trạng thái trong Q: {len(Q_table):4d} | X Thắng: {stats_x_wins:4d} | O Thắng: {stats_o_wins:4d} | Hòa: {stats_ties:4d}")
        stats_x_wins, stats_o_wins, stats_ties = 0, 0, 0

total_time = time.time() - t_start
print(f"\n🎉 HOÀN THÀNH HUẤN LUYỆN {max_episode} VÁN TRONG {total_time:.2f} GIÂY!")
print(f"📊 Tổng số trạng thái bàn cờ đã được khám phá và tối ưu: {len(Q_table)} trạng thái.")

🚀 BẮT ĐẦU HUẤN LUYỆN Q-LEARNING QUA 100000 VÁN TỰ ĐẤU (SELF-PLAY)...
Episode  2000/100000 (  0.4s) | Epsilon: 0.884 | Số trạng thái trong Q: 4520 | X Thắng: 1150 | O Thắng:  613 | Hòa:  237
Episode  4000/100000 (  0.7s) | Epsilon: 0.868 | Số trạng thái trong Q: 4520 | X Thắng: 1163 | O Thắng:  594 | Hòa:  243
Episode  6000/100000 (  1.0s) | Epsilon: 0.852 | Số trạng thái trong Q: 4520 | X Thắng: 1112 | O Thắng:  612 | Hòa:  276
Episode  8000/100000 (  1.3s) | Epsilon: 0.836 | Số trạng thái trong Q: 4520 | X Thắng: 1147 | O Thắng:  587 | Hòa:  266
Episode 10000/100000 (  1.5s) | Epsilon: 0.820 | Số trạng thái trong Q: 4520 | X Thắng: 1111 | O Thắng:  623 | Hòa:  266
Episode 12000/100000 (  1.8s) | Epsilon: 0.804 | Số trạng thái trong Q: 4520 | X Thắng: 1149 | O Thắng:  603 | Hòa:  248
Episode 14000/100000 (  2.0s) | Epsilon: 0.788 | Số trạng thái trong Q: 4520 | X Thắng: 1131 | O Thắng:  611 | Hòa:  258
Episode 16000/100000 (  2.3s) | Epsilon: 0.772 | Số trạng thái trong Q: 4520 | X Thắ

Lưu lại các model 1 theo số lượng episode huấn luyện khác nhau

In [47]:
# =============================================================================
# 6. LƯU VÀ TẢI BẢNG Q-TABLE TỪ FILE
# =============================================================================
import os
import pickle

SAVE_DIR = "files"
os.makedirs(SAVE_DIR, exist_ok=True)
Q_TABLE_FILE = os.path.join(SAVE_DIR, "q_table_ttt_100000.p")

# Lưu Q-table ra file
with open(Q_TABLE_FILE, "wb") as fp:
    pickle.dump(Q_table, fp)
print(f"💾 Đã lưu Q-table thành công tại: {Q_TABLE_FILE} ({os.path.getsize(Q_TABLE_FILE)/1024:.1f} KB)")

def load_q_table(filepath):
    """Hàm tải lại bảng Q-table từ đĩa."""
    global Q_table
    with open(filepath, "rb") as fp:
        Q_table = pickle.load(fp)
    print(f"📂 Đã nạp thành công Q-table ({len(Q_table)} trạng thái) từ {filepath}!")

💾 Đã lưu Q-table thành công tại: files\q_table_ttt_100000.p (900.6 KB)


# Kiểm thử mô hình Q-learning

Cho Q-learning đánh với Pure MCTS

In [49]:
# =============================================================================
# 7. ĐẤU TRƯỜNG ĐỐI ĐẦU: Q-LEARNING AGENT VS PURE MCTS (100 VÁN)
# =============================================================================

# --- THI ĐẤU 100 VÁN ---
total_matches = 1000
half = total_matches // 2
q_wins, ties, mcts_wins = 0, 0, 0

print(f"⚔️ Bắt đầu loạt trận đối đầu ({total_matches} ván)...")

# 50 ván Q-Agent đi trước (X)
for _ in range(half):
    g_env = ttt()
    while True:
        # Lượt 1: Q-Agent (X)
        act = q_agent_move(g_env)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == 1: q_wins += 1
            else: ties += 1
            break
            
        # Lượt 2: Pure MCTS (O)
        act = pure_mcts_move(g_env, num_rollouts=60)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == -1: mcts_wins += 1
            else: ties += 1
            break

# 50 ván Q-Agent đi sau (O)
for _ in range(half):
    g_env = ttt()
    while True:
        # Lượt 1: Pure MCTS (X)
        act = pure_mcts_move(g_env, num_rollouts=60)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == 1: mcts_wins += 1
            else: ties += 1
            break
            
        # Lượt 2: Q-Agent (O)
        act = q_agent_move(g_env)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == -1: q_wins += 1
            else: ties += 1
            break

# --- TỔNG KẾT BẢNG SỐ LIỆU ---
df_results = pd.DataFrame([{
    "Tổng số ván": total_matches,
    "Q-Agent Thắng": q_wins,
    "Hòa": ties,
    "Pure MCTS Thắng": mcts_wins,
    "Tỷ lệ Q Thắng (%)": f"{q_wins/total_matches*100:.1f}%",
    "Tỷ lệ Hòa (%)": f"{ties/total_matches*100:.1f}%",
    "Tỷ lệ Bất bại của Q (%)": f"{(q_wins + ties)/total_matches*100:.1f}%"
}])

print("\n📊 BẢNG TỔNG KẾT ĐỐI ĐẦU: Q-LEARNING VS PURE MCTS:")
display(df_results)

⚔️ Bắt đầu loạt trận đối đầu (1000 ván)...

📊 BẢNG TỔNG KẾT ĐỐI ĐẦU: Q-LEARNING VS PURE MCTS:


,Tổng số ván,Q-Agent Thắng,Hòa,Pure MCTS Thắng,Tỷ lệ Q Thắng (%),Tỷ lệ Hòa (%),Tỷ lệ Bất bại của Q (%)
0,1000,503,497,0,50.3%,49.7%,100.0%
